# BhramGuard Text Model Training

Train a phishing classifier for message or email text from `backend/ml/datasets/texts.csv`.

This notebook starts with a classical NLP baseline: TF-IDF word features plus handcrafted phishing-language signals. It saves model artifacts into `backend/ml/models`.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[2]

DATASET_PATH = PROJECT_ROOT / "backend" / "ml" / "datasets" / "texts.csv"
TEST_DATASET_PATH = PROJECT_ROOT / "backend" / "ml" / "datasets" / "phishing_legitimate_dataset.csv"
MODEL_DIR = PROJECT_ROOT / "backend" / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)
print("Optional test dataset:", TEST_DATASET_PATH)
print("Models:", MODEL_DIR)

In [ ]:
import re

import joblib
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

In [ ]:
df_text = pd.read_csv(DATASET_PATH)

print("Shape:", df_text.shape)
print("Columns:", df_text.columns.tolist())
df_text.head()

In [ ]:
def infer_columns(df):
    label_candidates = ["label", "Label", "class", "target", "is_phishing"]
    text_candidates = ["text", "message", "email", "body", "content"]

    label_col = next((col for col in label_candidates if col in df.columns), None)
    if label_col is None:
        raise ValueError(f"No label column found. Columns: {df.columns.tolist()}")

    text_col = next((col for col in text_candidates if col in df.columns and col != label_col), None)
    if text_col is None:
        object_cols = [col for col in df.columns if col != label_col and df[col].dtype == "object"]
        if not object_cols:
            raise ValueError(f"No text column found. Columns: {df.columns.tolist()}")
        text_col = object_cols[0]

    return text_col, label_col

TEXT_COL, LABEL_COL = infer_columns(df_text)
print("Text column:", TEXT_COL)
print("Label column:", LABEL_COL)
print(df_text[LABEL_COL].value_counts(dropna=False))

In [ ]:
df_text = df_text[[TEXT_COL, LABEL_COL]].dropna()
df_text[TEXT_COL] = df_text[TEXT_COL].astype(str)
df_text[LABEL_COL] = df_text[LABEL_COL].astype(int)
df_text = df_text.drop_duplicates(subset=[TEXT_COL])

print("Cleaned shape:", df_text.shape)
print(df_text[LABEL_COL].value_counts(normalize=True))

In [ ]:
def extract_text_features(value):
    text = "" if pd.isna(value) else str(value)
    lowered = text.lower()
    words = text.split()

    urgency_words = ["hurry", "urgent", "immediately", "act now", "limited time", "expire", "act fast"]
    reward_words = ["free", "win", "winner", "prize", "discount", "offer", "guarantee", "cash"]
    threat_words = ["suspend", "verify", "confirm", "account", "password", "security", "unauthorized"]
    authority_words = ["bank", "paypal", "microsoft", "google", "admin", "support", "team", "irs", "tax"]
    caps_words = [word for word in words if word.isupper() and len(word) > 1]

    return {
        "char_length": len(text),
        "word_count": len(words),
        "urgency_count": sum(1 for word in urgency_words if word in lowered),
        "reward_count": sum(1 for word in reward_words if word in lowered),
        "threat_count": sum(1 for word in threat_words if word in lowered),
        "authority_count": sum(1 for word in authority_words if word in lowered),
        "caps_ratio": len(caps_words) / len(words) if words else 0,
        "exclamation_count": text.count("!"),
        "question_count": text.count("?"),
        "link_count": len(re.findall(r"https?://|www\.", lowered)),
        "email_count": len(re.findall(r"[\w.%-]+@[\w.-]+\.[A-Za-z]{2,}", text)),
        "money_count": len(re.findall(r"[$]\s?\d+|\d+\s?(usd|dollars|rs|npr)", lowered)),
        "elongation_count": len(re.findall(r"([a-zA-Z])\1{3,}", text)),
    }

manual_features = pd.DataFrame(df_text[TEXT_COL].apply(extract_text_features).tolist())
manual_features.describe().T

In [ ]:
X_text_train, X_text_test, X_manual_train, X_manual_test, y_train, y_test = train_test_split(
    df_text[TEXT_COL],
    manual_features,
    df_text[LABEL_COL],
    test_size=0.2,
    random_state=42,
    stratify=df_text[LABEL_COL],
)

tfidf = TfidfVectorizer(
    max_features=5000,
    analyzer="word",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
    stop_words="english",
)

X_train_tfidf = tfidf.fit_transform(X_text_train)
X_test_tfidf = tfidf.transform(X_text_test)

X_train = hstack([X_train_tfidf, csr_matrix(X_manual_train.values)])
X_test = hstack([X_test_tfidf, csr_matrix(X_manual_test.values)])

print("Train matrix:", X_train.shape)
print("Test matrix:", X_test.shape)

In [ ]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "linear_svc": LinearSVC(class_weight="balanced", random_state=42),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced", n_jobs=-1),
}

results = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    auc = roc_auc_score(y_test, probabilities) if probabilities is not None else None

    print("\n===", model_name, "===")
    print(classification_report(y_test, predictions))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))
    if auc is not None:
        print("ROC-AUC:", auc)

    report = classification_report(y_test, predictions, output_dict=True)
    results[model_name] = {"model": model, "auc": auc, "f1": report["weighted avg"]["f1-score"]}

best_model_name = max(results, key=lambda name: (results[name]["auc"] or 0, results[name]["f1"]))
best_model = results[best_model_name]["model"]
print("Best model:", best_model_name)

In [ ]:
if TEST_DATASET_PATH.exists():
    df_external = pd.read_csv(TEST_DATASET_PATH).dropna()
    external_text_col, external_label_col = infer_columns(df_external)
    df_external[external_text_col] = df_external[external_text_col].astype(str)
    df_external[external_label_col] = df_external[external_label_col].astype(int)

    external_manual = pd.DataFrame(df_external[external_text_col].apply(extract_text_features).tolist())
    external_tfidf = tfidf.transform(df_external[external_text_col])
    external_X = hstack([external_tfidf, csr_matrix(external_manual.values)])
    external_y = df_external[external_label_col]
    external_pred = best_model.predict(external_X)

    print("External test dataset:", TEST_DATASET_PATH)
    print(classification_report(external_y, external_pred))
    print("Confusion matrix:")
    print(confusion_matrix(external_y, external_pred))

In [ ]:
artifact = {
    "model": best_model,
    "vectorizer": tfidf,
    "manual_feature_columns": manual_features.columns.tolist(),
    "text_column": TEXT_COL,
    "label_column": LABEL_COL,
    "best_model_name": best_model_name,
}

joblib.dump(artifact, MODEL_DIR / "text_phishing_model.pkl")
print("Saved:", MODEL_DIR / "text_phishing_model.pkl")